
<div style="background-color:#1F3864; padding:16px; border-radius:4px;">
<h1 style="color:#FFFFFF; font-family:Arial; margin:0;">Day 212 — Week 4, Day 1: Capstone Ideation & Scope</h1>
<p style="color:#D9E2F3; font-family:Arial; margin:4px 0 0 0;">Month 13 · Agentic AI &amp; Advanced GenAI Portfolio · TeleServe India Support Chatbot</p>
</div>

**Where this sits:** Days 200–211 built three separate, working components. Today locks how they
combine into ONE deployable artifact for Days 213–216, and — critically — how big that artifact
should be, given a hard 4-day build window.

- **LangGraph agent** (Days 200–203)
- **MCP tool** wrapping ResumeGapAI + the fileno/thread-bridge fixes (Days 204–207)
- **Fine-tuned DistilBERT ticket-priority classifier + inference wrapper** (Days 208–211)

This becomes the **3rd standalone starred GitHub repo**, closing the Month 12 exit criterion at
3/3 (currently 1/3 — only `ResumeGapAI` is standalone; `Month12-Capstone-Portfolio` and
`Month13-...` are umbrella repos, not separately star-seeded).



<div style="background-color:#1F3864; padding:10px; border-radius:4px;">
<h2 style="color:#FFFFFF; font-family:Arial; margin:0;">Section 1 — Raw Data (do not modify)</h2>
</div>

Unlike Day 197, the **project** isn't up for debate — the capstone chatbot was locked back on
Day 196/206. What's undecided is **scope**: how much of it to build in 4 days without overbuilding
(Day 197's Concept Note 5: *"Effort ≠ stars... overbuilding inside a fixed window is a bigger risk
than underbuilding"* — same principle, tighter window this time).

Four scope variants are scouted below, each scored on the same five attributes used in Day 197's
formula. `real_pain_evidence`/`personal_expertise_fit` are 1–5 self-assessed; `similar_repos_seen`
and `demo_seconds` are from a quick GitHub/README scan; `build_days_est` is an honest engineering
estimate against the actual 4-day (213–216) budget.


In [1]:

import pandas as pd

# Raw data: candidate SCOPE variants for the same locked project (support chatbot).
# Columns match Day 197's formula inputs exactly, so the same scoring function applies.
raw_data = pd.DataFrame([
    {
        "scope_id": "S1",
        "name": "Minimal Chatbot",
        "description": "LangGraph agent + Day211 classifier wrapper only. No MCP tool call, "
                        "single-turn, CLI or bare Streamlit text box.",
        "real_pain_evidence": 2,     # doesn't demo the MCP/agentic differentiator at all
        "personal_expertise_fit": 5, # already fully built, near-zero new risk
        "similar_repos_seen": 40,    # generic "chatbot + sklearn classifier" repos are common
        "demo_seconds": 10,
        "build_days_est": 1.5,
    },
    {
        "scope_id": "S2",
        "name": "Core Integration",
        "description": "LangGraph agent routes a ticket -> calls Day211 classifier as a tool -> "
                        "calls the Day204 MCP tool for a second signal -> single clean Streamlit "
                        "UI showing priority + confidence + routing decision.",
        "real_pain_evidence": 4,     # directly demos agent+MCP+fine-tuned-model working together
        "personal_expertise_fit": 5, # every piece is already built and verified in isolation
        "similar_repos_seen": 6,     # agent+MCP+custom fine-tuned model combos are uncommon
        "demo_seconds": 20,
        "build_days_est": 3.0,
    },
    {
        "scope_id": "S3",
        "name": "Core + Human-in-the-Loop",
        "description": "Everything in S2, plus the confidence<0.60 human-review lane (Day211's real "
                        "threshold) surfaced as a distinct UI state, plus a small session-memory "
                        "layer so the agent recalls prior tickets in one sitting.",
        "real_pain_evidence": 5,     # the human-review lane is the actual production-relevant idea
        "personal_expertise_fit": 4, # session memory in LangGraph is new-ish ground for this track
        "similar_repos_seen": 3,
        "demo_seconds": 30,
        "build_days_est": 4.0,
    },
    {
        "scope_id": "S4",
        "name": "Enterprise Extended",
        "description": "Everything in S3, plus CrewAI multi-agent delegation (Day206-207) and a "
                        "vector-DB ticket-similarity lookup layered on top.",
        "real_pain_evidence": 4,     # impressive but dilutes the clean triage story
        "personal_expertise_fit": 3, # vector DB integration is genuinely new, not yet practiced
        "similar_repos_seen": 2,
        "demo_seconds": 45,
        "build_days_est": 6.5,       # exceeds the 4-day window on its own estimate
    },
])

print(raw_data.shape)
raw_data


(4, 8)


,scope_id,name,description,real_pain_evidence,personal_expertise_fit,similar_repos_seen,demo_seconds,build_days_est
0,S1,Minimal Chatbot,LangGraph agent + Day211 classifier wrapper on...,2,5,40,10,1.5
1,S2,Core Integration,LangGraph agent routes a ticket -> calls Day21...,4,5,6,20,3.0
2,S3,Core + Human-in-the-Loop,"Everything in S2, plus the confidence<0.60 hum...",5,4,3,30,4.0
3,S4,Enterprise Extended,"Everything in S3, plus CrewAI multi-agent dele...",4,3,2,45,6.5



<div style="background-color:#1F3864; padding:10px; border-radius:4px;">
<h2 style="color:#FFFFFF; font-family:Arial; margin:0;">Section 2 — Concept Notes</h2>
</div>

**1. Scoping is a distinct skill from building.** Days 208–211 proved every individual piece
works. Today's risk isn't "can I build this" — it's "will I finish the RIGHT version of this in
4 days." That's a scope-management skill, and it's exactly what clients pay a freelancer to have:
knowing what to cut.

**2. Reusing Day 197's formula on scope, not project choice.** The formula doesn't care whether the
candidates are different projects or different sizes of the same project — it's scoring
*pain evidence, personal fit, novelty, demo speed, effort* either way:

```
score = (real_pain_evidence * 3)
      + (personal_expertise_fit * 2)
      + (max(0, 5 - similar_repos_seen // 3))     # novelty bucket, capped 0-5
      + (max(0, 5 - demo_seconds // 5))           # demo-speed bucket, capped 0-5
      - (build_days_est * 1.5)                    # effort penalty
```

**3. `build_days_est` is a hard constraint here, not just a penalty term.** Day 197 treated effort
as a soft negative weight. Here there's a literal ceiling: only Days 213–216 exist (4 days) before
Day 216's launch checklist. A scope whose own `build_days_est` exceeds 4.0 should be flagged as
disqualified regardless of score, not just penalized.

**4. Differentiation applies against your OWN portfolio, twice now.** Day 197 checked the new
project against TeleServe (RAG+ML+Streamlit). Today's check is against *two* prior repos:
TeleServe (RAG/retrieval) and ResumeGapAI (similarity scoring). The chatbot must not just be
"TeleServe's classifier with a chat window bolted on" — the MCP tool call and the agent's routing
logic are what make it a structurally different artifact, not just a different UI.

**5. Star-worthiness ≠ feature count.** S4 (Enterprise Extended) looks the most impressive on paper
but is the weakest real submission: it can't be finished cleanly in the window, and an unfinished
"impressive" repo reads worse to a hiring client than a finished "focused" one. A GitHub visitor
(or interviewer) rewards something that clearly works end-to-end over something with more moving
parts that's visibly rushed.

**6. Freelance framing matters for the choice too.** Per the standing Upwork-positioning note
(chatbot-shaped deliverables +71% YoY, "AI integration" +178% YoY), the scope that best supports
a freelance pitch is one with a clean before/after story: *"ticket comes in -> agent + fine-tuned
model + external tool -> routed decision."* That story needs to be demoable in under 30 seconds —
which is itself one of the scoring inputs (`demo_seconds`).



<div style="background-color:#1F3864; padding:10px; border-radius:4px;">
<h2 style="color:#FFFFFF; font-family:Arial; margin:0;">Section 3 — Practice Guide</h2>
</div>

Complete Tasks 1–5 below in order. NRA rule applies: every number in Task 5 must come from this
run's `print()` output, not memory or the answer key.



**Task 1 — Inspect the raw scope candidates**
Print `raw_data.shape`, then print `scope_id`, `name`, and `build_days_est` sorted by
`build_days_est` ascending. Pure inspection — no scoring yet.


In [2]:
# ── TASK 1: Inspect raw scope candidates ─────────────────────────────
# Goal: Print the shape of the raw data and list the candidates sorted
#       by estimated build time (shortest first) to get an overview.
# Method: Use raw_data.shape, then sort by 'build_days_est' ascending
#         and print scope_id, name, and build_days_est.

print("Raw data shape:", raw_data.shape)
print("\nCandidates sorted by build_days_est (ascending):")
sorted_by_build = raw_data.sort_values("build_days_est")
print(sorted_by_build[["scope_id", "name", "build_days_est"]].to_string(index=False))

Raw data shape: (4, 8)

Candidates sorted by build_days_est (ascending):
scope_id                     name  build_days_est
      S1          Minimal Chatbot             1.5
      S2         Core Integration             3.0
      S3 Core + Human-in-the-Loop             4.0
      S4      Enterprise Extended             6.5



**Task 2 — Apply the star-worthiness formula, with a hard budget disqualifier**
Write `score_candidate(row)` implementing the exact Day 197 formula from Concept Note 2. Apply it
to every row, storing the result in `star_score`. Then add a boolean column `over_budget` that is
`True` when `build_days_est > 4.0` (the real Days 213–216 window).


In [3]:
# ── TASK 2: Scoring function + hard budget flag ──────────────────────
# Goal: Implement the exact Day197 formula and flag any candidate whose
#       build estimate exceeds the 4‑day window.
# Method: Define score_candidate(row) using the formula:
#         score = real_pain_evidence*3 + personal_expertise_fit*2
#                 + max(0, 5 - similar_repos_seen//3)
#                 + max(0, 5 - demo_seconds//5)
#                 - build_days_est*1.5
#         Then apply to each row and add an 'over_budget' column.

def score_candidate(row):
    pain = row["real_pain_evidence"] * 3
    expertise = row["personal_expertise_fit"] * 2
    novelty = max(0, 5 - row["similar_repos_seen"] // 3)
    demo_speed = max(0, 5 - row["demo_seconds"] // 5)
    effort_penalty = row["build_days_est"] * 1.5
    return pain + expertise + novelty + demo_speed - effort_penalty

# Apply scoring
raw_data["star_score"] = raw_data.apply(score_candidate, axis=1)

# Budget disqualifier: build_days_est > 4.0 (the real Days 213–216 window)
raw_data["over_budget"] = raw_data["build_days_est"] > 4.0

print("Scored candidates:")
print(raw_data[["scope_id", "name", "star_score", "over_budget", "build_days_est"]].to_string(index=False))

Scored candidates:
scope_id                     name  star_score  over_budget  build_days_est
      S1          Minimal Chatbot       16.75        False             1.5
      S2         Core Integration       21.50        False             3.0
      S3 Core + Human-in-the-Loop       21.00        False             4.0
      S4      Enterprise Extended       13.25         True             6.5



**Task 3 — Rank, then apply the disqualifier**
Sort by `star_score` descending and print the full ranked table. Then print the ranked table
again with `over_budget == True` rows excluded, and store that filtered, ranked frame as
`eligible`. State in a comment which scope tops each of the two rankings.


In [4]:
# ── TASK 3: Rank and disqualify ──────────────────────────────────────
# Goal: Sort all candidates by star_score descending, then exclude the
#       over‑budget candidates and print the eligible ranking.
# Method: Sort raw_data by star_score descending, store as ranked_full.
#         Then filter out over_budget == True, sort again, store as eligible.

# Full ranking (including disqualified)
ranked_full = raw_data.sort_values("star_score", ascending=False)
print("=== Full ranking (including over‑budget candidates) ===")
print(ranked_full[["scope_id", "name", "star_score", "over_budget", "build_days_est"]].to_string(index=False))

# Eligible candidates (over_budget == False) ranked by score
eligible = raw_data[~raw_data["over_budget"]].sort_values("star_score", ascending=False)
print("\n=== Eligible ranking (over‑budget removed) ===")
print(eligible[["scope_id", "name", "star_score", "build_days_est"]].to_string(index=False))

# Which scope tops each ranking?
print("\nTop in full ranking: S2 (Core Integration) with score 21.50")
print("Top in eligible ranking: S2 (Core Integration) with score 21.50")
print("(The disqualifier removed S4 but did not change the top choice.)")

=== Full ranking (including over‑budget candidates) ===
scope_id                     name  star_score  over_budget  build_days_est
      S2         Core Integration       21.50        False             3.0
      S3 Core + Human-in-the-Loop       21.00        False             4.0
      S1          Minimal Chatbot       16.75        False             1.5
      S4      Enterprise Extended       13.25         True             6.5

=== Eligible ranking (over‑budget removed) ===
scope_id                     name  star_score  build_days_est
      S2         Core Integration       21.50             3.0
      S3 Core + Human-in-the-Loop       21.00             4.0
      S1          Minimal Chatbot       16.75             1.5

Top in full ranking: S2 (Core Integration) with score 21.50
Top in eligible ranking: S2 (Core Integration) with score 21.50
(The disqualifier removed S4 but did not change the top choice.)



**Task 4 — Differentiation check**
Write a short (3–5 line) printed check comparing the winning eligible scope against BOTH prior
repos: TeleServe capstone (RAG + ML + Streamlit over ticket text) and ResumeGapAI (TF-IDF/embedding
similarity scoring). State concretely what structural element makes the winner distinct from each
— not a restatement that it "uses different tech."


In [5]:
# ── TASK 4: Differentiation check ────────────────────────────────────
# Goal: Confirm the winning scope (S2) is structurally distinct from
#       both prior repos (TeleServe and ResumeGapAI) to avoid portfolio
#       redundancy.
# Method: Print a concrete architectural difference for each prior repo.

winner = eligible.iloc[0]
print(f"Winning scope: {winner['name']} (S2)")
print("\n--- Distinction vs TeleServe capstone ---")
print("- TeleServe: RAG + vector DB retrieval over tickets + Streamlit UI, no agentic flow, no fine‑tuned classifier.")
print("- S2: Uses a LangGraph agent that routes a ticket through two tools (the fine‑tuned DistilBERT classifier and the MCP tool) in a single conversational turn, producing a structured decision with confidence. This is an agent‑centric workflow, not a retrieval‑first pipeline.")
print("\n--- Distinction vs ResumeGapAI ---")
print("- ResumeGapAI: Similarity scoring (TF‑IDF/embeddings) over text pairs, no multi‑step logic, no model fine‑tuning.")
print("- S2: Combines a fine‑tuned classifier (Day211), an external MCP tool call (Day204), and a LangGraph decision node – all orchestrated as an agent. The output is not a similarity score but a priority + routing decision, which is a different product category.")

Winning scope: Core Integration (S2)

--- Distinction vs TeleServe capstone ---
- TeleServe: RAG + vector DB retrieval over tickets + Streamlit UI, no agentic flow, no fine‑tuned classifier.
- S2: Uses a LangGraph agent that routes a ticket through two tools (the fine‑tuned DistilBERT classifier and the MCP tool) in a single conversational turn, producing a structured decision with confidence. This is an agent‑centric workflow, not a retrieval‑first pipeline.

--- Distinction vs ResumeGapAI ---
- ResumeGapAI: Similarity scoring (TF‑IDF/embeddings) over text pairs, no multi‑step logic, no model fine‑tuning.
- S2: Combines a fine‑tuned classifier (Day211), an external MCP tool call (Day204), and a LangGraph decision node – all orchestrated as an agent. The output is not a similarity score but a priority + routing decision, which is a different product category.



**Task 5 — Locked scope decision (NRA)**
Using the eligible-ranking output from Task 3, write the final decision as Number / Reason /
Action. Number = the actual winning `star_score` from `eligible`. Reason = the mechanism (why it
won among eligible candidates — reference the budget disqualifier if it changed the outcome vs the
unfiltered ranking). Action = a locked day-by-day plan for Days 213–216 naming concrete deliverables
per day, not vague goals.


In [7]:
# ── TASK 5: NRA final decision ──────────────────────────────────────
# Goal: Write the final choice as Number / Reason / Action, with
#       the Number sourced from the live eligible ranking.
# Method: Extract star_score from the top eligible row; craft a Reason
#         that explains the mechanism, including the effect of the
#         budget disqualifier; provide a day‑by‑day plan.

# Number: star_score of the winning eligible candidate
winning_score = eligible.iloc[0]["star_score"]
winning_id = eligible.iloc[0]["scope_id"]
winning_name = eligible.iloc[0]["name"]

print("=== NRA Final Decision ===")
print(f"Number: {winning_score}")
print("Reason: S2 (Core Integration) won with the highest star_score (21.50) among eligible candidates. Its score reflects strong pain evidence (4), full personal expertise (5), moderate novelty (3), and a fast demo (20s), while its build estimate (3.0 days) leaves a 1‑day buffer within the 4‑day window. S3 scored close (21.00) but had lower expertise (4) and exactly filled the budget, leaving zero buffer for unexpected issues. S4 was both the lowest scorer (13.25) and over budget (6.5 days), so it was never competitive; the disqualifier simply confirmed it was already out on merit.")
print("Action: Lock S2 as the build scope. Day‑by‑day plan for Days 213–216:")
print("  - Day 213: Assemble the LangGraph agent skeleton with the two tool nodes (classifier wrapper and MCP tool) and a simple router that prints intermediate outputs. (No UI yet.)")
print("  - Day 214: Connect the actual Day211 wrapper and MCP tool functions, test end‑to‑end on 10 sample tickets, fix any integration bugs.")
print("  - Day 215: Build the Streamlit interface: text input, display of predicted priority + confidence + routing decision, and the MCP tool's secondary signal (if any).")
print("  - Day 216: Polish UI (error handling, loading states), write a brief README with a demo GIF or screenshots, and freeze the repo. No new features – only bug fixes and documentation.")

=== NRA Final Decision ===
Number: 21.5
Reason: S2 (Core Integration) won with the highest star_score (21.50) among eligible candidates. Its score reflects strong pain evidence (4), full personal expertise (5), moderate novelty (3), and a fast demo (20s), while its build estimate (3.0 days) leaves a 1‑day buffer within the 4‑day window. S3 scored close (21.00) but had lower expertise (4) and exactly filled the budget, leaving zero buffer for unexpected issues. S4 was both the lowest scorer (13.25) and over budget (6.5 days), so it was never competitive; the disqualifier simply confirmed it was already out on merit.
Action: Lock S2 as the build scope. Day‑by‑day plan for Days 213–216:
  - Day 213: Assemble the LangGraph agent skeleton with the two tool nodes (classifier wrapper and MCP tool) and a simple router that prints intermediate outputs. (No UI yet.)
  - Day 214: Connect the actual Day211 wrapper and MCP tool functions, test end‑to‑end on 10 sample tickets, fix any integration bu


<div style="background-color:#1F3864; padding:10px; border-radius:4px;">
<h2 style="color:#FFFFFF; font-family:Arial; margin:0;">Section 4 — Scoring Rubric</h2>
</div>

| Task | Points | Criteria |
|---|---|---|
| Task 1 — Inspect | 10 | Correct shape printed; sort by `build_days_est` ascending correct |
| Task 2 — Scoring fn + budget flag | 25 | Formula matches Day 197 exactly (weights, bucket caps); `over_budget` threshold correct (>4.0) |
| Task 3 — Rank + disqualify | 20 | Full ranking correct; `eligible` correctly excludes over-budget row(s); both printed |
| Task 4 — Differentiation check | 15 | Concrete structural distinction named for BOTH prior repos, not a generic "different tech" claim |
| Task 5 — NRA decision | 30 | Number from live `eligible` output (not memory); Reason states the mechanism incl. disqualifier's role; Action is a named, dated, concrete 4-day plan |
| **Total** | **100** | +10★ bonus available for a clean single-submission pass (no resubmit) |

**Technical vs. Communication split (flagged separately per program rule 6):**
- *Technical* failures: wrong formula weights/buckets, wrong budget threshold, `eligible` filter
  applied in the wrong direction, Number not sourced from live output.
- *Communication* failures: Task 4 restating "uses different libraries" instead of naming the
  structural mechanism; Task 5 Reason bleeding into Action (restating the plan instead of the
  cause); Action written as vague goals ("polish the UI") instead of named deliverables per day.



<div style="background-color:#1F3864; padding:10px; border-radius:4px;">
<h2 style="color:#FFFFFF; font-family:Arial; margin:0;">Interview Framing</h2>
</div>

**"Walk me through how you scoped this project."**

*"I had three working components — a LangGraph agent, an MCP tool integration, and a fine-tuned
DistilBERT classifier with a real precision/recall profile from live evaluation. Rather than combine
all of them plus extras into one build, I scored four scope variants on pain-evidence, my own
expertise fit, novelty versus existing repos, demo speed, and estimated build time — then applied
a hard disqualifier for anything that wouldn't fit the actual time budget, even if it scored well on
paper. The most 'impressive'-looking option — multi-agent delegation plus a vector DB — got cut
because it couldn't be finished cleanly in four days, and a half-finished ambitious repo hurts a
portfolio more than a finished focused one. I locked a scope that's fully buildable in the window
and still demonstrates the differentiator that matters: an agent making a live tool call and a
model call mid-conversation, not just a chat UI in front of a static script."*

This is the answer that shows scope discipline, not just technical range — the trait clients
actually pay for.
